# Implementation: Impenetrable Barrier at L ≈ 2.8 / 구현: L ≈ 2.8의 침투 불가 장벽

**Paper / 논문**: Baker, Jaynes, Hoxie, Thorne et al., *Nature* **515**, 531–534 (2014).

**English**
This notebook reproduces three key findings of Baker et al. (2014):
1. The plasmapause location L_pp(Kp) and its co-location with the >2 MeV electron inner edge near L ≈ 2.8.
2. The radial profile of ultrarelativistic (>2 MeV) electrons before and after the persistent barrier formed in 2013.
3. The competition between plasmaspheric hiss damping (continuous, broad) and EMIC wave damping (episodic, narrow) near the plasmapause.

**한국어**
이 노트북은 Baker 외(2014)의 세 가지 핵심 결과를 재현한다:
1. 플라즈마권계면 위치 L_pp(Kp)와 >2 MeV 전자 내측 경계의 L ≈ 2.8 공동 위치.
2. 2013년 barrier 형성 전후 초상대론적(>2 MeV) 전자의 방사형 분포.
3. 플라즈마권계면 근처에서 plasmaspheric hiss 감쇠(연속·광범위)와 EMIC 파동 감쇠(간헐·협소)의 경쟁.

## Part 1: Setup / 설정

**English** Import scientific Python stack and define physical constants used throughout.

**한국어** 과학용 파이썬 라이브러리를 import하고, 전체에서 사용하는 물리 상수를 정의한다.

In [ ]:
"""Setup and shared physical constants.

All wave-particle and diffusion calculations below assume a dipole field model.
"""

import numpy as np
import matplotlib.pyplot as plt

# Physical constants (SI).
C = 2.998e8                # Speed of light, m/s.
ME = 9.109e-31             # Electron rest mass, kg.
QE = 1.602e-19             # Elementary charge, C.
MEC2_MEV = 0.511           # Electron rest energy, MeV.
B0_EARTH_NT = 31000.0      # Surface equatorial dipole field, nT.
RE_KM = 6371.0             # Earth radius, km.

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Setup complete: constants loaded.")

## Part 2: Plasmapause Location L_pp(Kp) / 플라즈마권계면 위치

**English**
We use the Carpenter & Anderson (1992) empirical relation
$L_{pp} = 5.6 - 0.46\,\mathrm{Kp_{max}}$, valid for daytime conditions and the average plasmapause position.

**한국어**
Carpenter & Anderson(1992)의 경험식 $L_{pp} = 5.6 - 0.46\,\mathrm{Kp_{max}}$을 사용한다. 주간 평균 플라즈마권계면 위치에 적용 가능.

In [ ]:
def plasmapause_L(kp_max: np.ndarray) -> np.ndarray:
    """Return plasmapause location L_pp from Carpenter & Anderson (1992).

    Args:
        kp_max: Past 24-hour maximum Kp index. Scalar or numpy array.

    Returns:
        Plasmapause radial distance in Earth radii (Re).
    """
    return 5.6 - 0.46 * np.asarray(kp_max, dtype=float)


# Evaluate over a Kp grid spanning quiet to extreme storm conditions.
kp_grid = np.linspace(0, 9, 100)
lpp_grid = plasmapause_L(kp_grid)

# Highlight key activity levels.
kp_examples = np.array([0, 2, 4, 6, 8])
lpp_examples = plasmapause_L(kp_examples)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kp_grid, lpp_grid, "b-", lw=2, label=r"$L_{pp} = 5.6 - 0.46\,K_{p,\max}$")
ax.scatter(kp_examples, lpp_examples, c="red", zorder=5)
for k, l in zip(kp_examples, lpp_examples):
    ax.annotate(f"Kp={k}\nL={l:.2f}", (k, l), textcoords="offset points", xytext=(8, 5))
ax.axhline(2.8, color="k", ls="--", alpha=0.6, label=r"Barrier $L=2.8$")
ax.set_xlabel(r"$K_{p,\max}$ (past 24 h)")
ax.set_ylabel(r"Plasmapause $L_{pp}$ ($R_E$)")
ax.set_title("Plasmapause Location vs. Geomagnetic Activity / 플라즈마권계면 vs. 지자기 활동도")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Quiet (Kp=2): L_pp = {plasmapause_L(2):.2f}")
print(f"Storm (Kp=6): L_pp = {plasmapause_L(6):.2f}  (matches barrier at L=2.84)")
print(f"Extreme (Kp=8): L_pp = {plasmapause_L(8):.2f}")

**Result / 결과**: For Kp ≈ 6, L_pp drops to ≈ 2.84 — exactly co-located with the observed >2 MeV inner edge. Even during extreme storms (Kp = 8), L_pp ≈ 1.92, but the barrier itself rarely retreats below L ≈ 2.6 because the *minimum-attained* L_pp determines the persistent edge.

**Kp ≈ 6에서 L_pp ≈ 2.84로 떨어져 관측된 >2 MeV 내측 경계와 정확히 공동 위치한다. 극심한 폭풍(Kp = 8)에도 L_pp ≈ 1.92이지만, *도달한 최소* L_pp가 지속적 경계를 결정하므로 barrier 자체는 L ≈ 2.6 아래로 거의 후퇴하지 않는다.**

## Part 3: Time-series Co-location of L_pp and the Barrier / L_pp와 Barrier의 시계열 공동 위치

**English**
We synthesize a 2-year Kp index time series mimicking realistic geomagnetic activity and compare the resulting L_pp(t) with a model >2 MeV electron inner-edge L_edge(t) that is bounded below by L = 2.8 except during the most extreme L_pp compressions.

**한국어**
현실적 지자기 활동도를 모사한 2년치 Kp 지수 시계열을 합성하고, 결과 L_pp(t)와 모델 >2 MeV 전자 내측 경계 L_edge(t)를 비교한다. 후자는 가장 극심한 L_pp 압축 시 외에는 L = 2.8을 하한으로 한다.

In [ ]:
def synthesize_kp(n_days: int, seed: int = 7) -> np.ndarray:
    """Generate a synthetic Kp time-series with storm-quiet patterns.

    Args:
        n_days: Number of days to simulate.
        seed: RNG seed for reproducibility.

    Returns:
        Array of daily-max Kp values with realistic skewness.
    """
    rs = np.random.default_rng(seed)
    base = 1.5 + 1.0 * np.sin(2 * np.pi * np.arange(n_days) / 27.0)  # 27-day recurrence.
    noise = rs.gamma(shape=2.0, scale=1.2, size=n_days)
    storms = rs.choice([0, 0, 0, 1], size=n_days)
    storm_amp = rs.uniform(2.0, 5.0, size=n_days) * storms
    kp = np.clip(base + 0.5 * noise + storm_amp, 0.0, 9.0)
    return kp


def barrier_inner_edge(lpp: np.ndarray, persistent_floor: float = 2.8) -> np.ndarray:
    """Model >2 MeV electron inner edge using minimum-attained L_pp.

    The barrier follows L_pp downward only briefly during severe compressions;
    otherwise it stays at the persistent floor near 2.8 R_E.

    Args:
        lpp: Time-series of plasmapause location.
        persistent_floor: Long-term innermost barrier location, R_E.

    Returns:
        Time-series of >2 MeV electron inner edge L_edge.
    """
    edge = np.maximum(persistent_floor, lpp - 0.05)
    # Allow brief inward incursions when L_pp drops well below the floor.
    extreme = lpp < (persistent_floor - 0.3)
    edge = np.where(extreme, lpp + 0.1, edge)
    return edge


n_days = 730
kp_series = synthesize_kp(n_days)
lpp_series = plasmapause_L(kp_series)
edge_series = barrier_inner_edge(lpp_series)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
t = np.arange(n_days)
axes[0].plot(t, kp_series, color="gray", lw=0.8)
axes[0].set_ylabel("Kp index")
axes[0].set_title("Synthetic Kp time series (2 years) / 합성 Kp 2년 시계열")
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, lpp_series, color="steelblue", lw=0.8, label=r"$L_{pp}$ (plasmapause)")
axes[1].plot(t, edge_series, color="crimson", lw=1.0, label=r">2 MeV inner edge $L_{\rm edge}$")
axes[1].axhline(2.8, color="k", ls="--", alpha=0.5)
axes[1].set_xlabel("Day from 1 Sep 2012 / 2012-09-01 기준 일수")
axes[1].set_ylabel(r"L (Earth radii)")
axes[1].invert_yaxis()
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

frac_at_floor = np.mean(np.isclose(edge_series, 2.8, atol=0.05))
print(f"Fraction of time edge sits at L≈2.8: {frac_at_floor:.1%}")
print(f"Min L_pp reached: {lpp_series.min():.2f}")
print(f"Min edge reached: {edge_series.min():.2f}")

## Part 4: Radial Profile Before & After Barrier Formation / Barrier 형성 전후 방사형 분포

**English**
Solve the steady-state 1D radial-diffusion equation with a localized hiss-loss term inside the plasmapause:
$$ L^{*2}\,\frac{d}{dL^*}\!\left(\frac{D_{LL}}{L^{*2}}\frac{df}{dL^*}\right) - \frac{f}{\tau_{\rm loss}(L^*)} = 0. $$
We use a finite-difference solver with `np.trapezoid` for diagnostic integration. Two cases are compared: pre-2013 (no strong hiss boundary, smooth profile) and post-2013 (sharp barrier inside L_pp).

**한국어**
플라즈마권계면 내부에 국소화된 hiss 손실 항을 포함한 정상 상태 1D 방사형 확산 방정식을 푼다. 진단 적분에는 `np.trapezoid`를 사용한다. 두 경우 비교: 2013년 이전(강한 hiss 경계 없음, 매끄러운 분포), 2013년 이후(L_pp 내부의 날카로운 barrier).

In [ ]:
def solve_radial_diffusion(
    l_grid: np.ndarray,
    d_ll: np.ndarray,
    inv_tau_loss: np.ndarray,
    f_outer: float = 1.0,
    f_inner: float = 0.0,
) -> np.ndarray:
    """Steady-state 1D radial diffusion with sink term.

    Solves L*^2 d/dL*( D_LL / L*^2 df/dL* ) - f / tau_loss = 0 by tridiagonal
    finite differencing on a uniform L grid.

    Args:
        l_grid: Uniform L* grid (R_E).
        d_ll: Diffusion coefficient at each grid node, day^-1.
        inv_tau_loss: Inverse loss timescale at each node, day^-1.
        f_outer: Dirichlet boundary condition at l_grid[-1].
        f_inner: Dirichlet boundary condition at l_grid[0].

    Returns:
        Array of f(L*) values on the grid.
    """
    n = l_grid.size
    h = l_grid[1] - l_grid[0]
    a = np.zeros(n)
    b = np.zeros(n)
    c = np.zeros(n)
    d = np.zeros(n)

    for i in range(1, n - 1):
        l = l_grid[i]
        # Effective coefficient k(L) = D_LL / L^2; equation in terms of (df/dL).
        k_minus = 0.5 * (d_ll[i] + d_ll[i - 1]) / (0.5 * (l_grid[i] + l_grid[i - 1])) ** 2
        k_plus = 0.5 * (d_ll[i] + d_ll[i + 1]) / (0.5 * (l_grid[i] + l_grid[i + 1])) ** 2
        coeff = l ** 2 / h ** 2
        a[i] = coeff * k_minus
        c[i] = coeff * k_plus
        b[i] = -(a[i] + c[i]) - inv_tau_loss[i]
        d[i] = 0.0
    # Dirichlet boundaries.
    b[0] = 1.0
    d[0] = f_inner
    b[-1] = 1.0
    d[-1] = f_outer

    # Thomas algorithm for tridiagonal system.
    cp = np.zeros(n)
    dp = np.zeros(n)
    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * cp[i - 1]
        cp[i] = c[i] / m if i < n - 1 else 0.0
        dp[i] = (d[i] - a[i] * dp[i - 1]) / m
    f = np.zeros(n)
    f[-1] = dp[-1]
    for i in range(n - 2, -1, -1):
        f[i] = dp[i] - cp[i] * f[i + 1]
    return f


# L grid and diffusion coefficient (Brautigam & Albert 2000 form, Kp ~ 4).
l_grid = np.linspace(2.0, 6.0, 401)
kp_assumed = 4.0
d_ll = 10.0 ** (0.506 * kp_assumed - 9.325) * l_grid ** 10  # 1/day.

# Pre-barrier scenario: weak hiss (assume hiss only above L=4 ineffective).
tau_loss_pre = np.full_like(l_grid, np.inf)
tau_loss_pre[l_grid < 2.5] = 30.0  # Slow atmospheric loss only.

# Post-barrier scenario: strong hiss inside plasmapause (L_pp = 2.8).
L_PP = 2.8
tau_loss_post = np.where(l_grid < L_PP, 3.0, np.inf)  # 3-day hiss lifetime.

f_pre = solve_radial_diffusion(l_grid, d_ll, 1.0 / tau_loss_pre, f_outer=1.0, f_inner=0.0)
f_post = solve_radial_diffusion(l_grid, d_ll, 1.0 / tau_loss_post, f_outer=1.0, f_inner=0.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(l_grid, np.clip(f_pre, 1e-6, None), "b-", lw=2, label="Pre-2013 (weak hiss) / 2013 이전")
ax.semilogy(l_grid, np.clip(f_post, 1e-6, None), "r-", lw=2, label="Post-2013 (strong hiss inside L_pp) / 2013 이후")
ax.axvline(L_PP, color="k", ls="--", alpha=0.6, label=f"$L_{{pp}}={L_PP}$")
ax.set_xlabel("$L^*$ (Earth radii)")
ax.set_ylabel("PSD f (arbitrary units)")
ax.set_title(">2 MeV Electron PSD Profile Before/After Barrier / Barrier 전후 PSD")
ax.set_ylim(1e-5, 2.0)
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Diagnostic: integrated PSD inside L_pp using np.trapezoid.
mask_inside = l_grid < L_PP
psd_inside_pre = np.trapezoid(f_pre[mask_inside], l_grid[mask_inside])
psd_inside_post = np.trapezoid(f_post[mask_inside], l_grid[mask_inside])
print(f"Integrated PSD inside L_pp (pre-2013):  {psd_inside_pre:.4e}")
print(f"Integrated PSD inside L_pp (post-2013): {psd_inside_post:.4e}")
print(f"Ratio (loss factor): {psd_inside_pre / max(psd_inside_post, 1e-20):.2e}")

## Part 5: Hiss vs. EMIC Damping near the Plasmapause / 플라즈마권계면 근처 hiss vs. EMIC 감쇠

**English**
Compare bounce-averaged pitch-angle diffusion coefficients $\langle D_{\alpha\alpha}\rangle$ for plasmaspheric hiss and EMIC waves as a function of L. We use simplified Summers-style scalings:
- Hiss: continuous, B_w ≈ 50 pT inside plasmasphere, ineffective outside.
- EMIC: episodic, B_w ≈ 1 nT in narrow L band (3.5 < L < 5), strong for >2 MeV.

**한국어**
플라즈마권계면 근처에서 plasmaspheric hiss와 EMIC 파동의 bounce-averaged 피치각 확산 계수를 L의 함수로 비교한다. 단순화된 Summers 스케일링 사용. Hiss는 플라즈마권 내부 연속적, B_w ≈ 50 pT, 외부에서 비효과적. EMIC는 좁은 L 대역(3.5 < L < 5)에 산발적, B_w ≈ 1 nT.

In [ ]:
def hiss_d_alpha(l: np.ndarray, energy_mev: float, l_pp: float = 2.8) -> np.ndarray:
    """Approximate bounce-averaged D_alpha_alpha for plasmaspheric hiss.

    Uses a B_w^2 / (gamma^2 * B0^2) scaling truncated outside the plasmasphere.

    Args:
        l: Array of L-shell values.
        energy_mev: Electron kinetic energy in MeV.
        l_pp: Plasmapause location, R_E.

    Returns:
        D_alpha_alpha in 1/day units.
    """
    bw_pt = 50.0
    b0_nt = B0_EARTH_NT / l ** 3
    gamma = 1.0 + energy_mev / MEC2_MEV
    norm = (bw_pt / 1000.0) ** 2 / (gamma ** 2 * b0_nt ** 2)
    base = norm * 5e10  # Empirical normalization to give ~1/3 day at L=2.8, 2 MeV.
    return np.where(l < l_pp, base, base * 1e-4)


def emic_d_alpha(l: np.ndarray, energy_mev: float) -> np.ndarray:
    """Approximate bounce-averaged D_alpha_alpha for EMIC waves.

    Strong for relativistic electrons (>2 MeV), confined to a duskside plume
    band 3.5 < L < 5; episodic occurrence factor 0.1.

    Args:
        l: Array of L-shell values.
        energy_mev: Electron kinetic energy in MeV.

    Returns:
        D_alpha_alpha in 1/day units (occurrence-averaged).
    """
    occurrence = 0.1
    threshold = 2.0
    energy_factor = max(0.0, energy_mev - threshold) / threshold
    in_band = (l > 3.5) & (l < 5.0)
    return np.where(in_band, occurrence * energy_factor * 0.5, 0.0)


l_grid_w = np.linspace(2.0, 6.0, 401)
energies = [1.0, 2.0, 4.0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for energy in energies:
    axes[0].semilogy(l_grid_w, hiss_d_alpha(l_grid_w, energy) + 1e-12,
                     label=f"E={energy} MeV")
    axes[1].semilogy(l_grid_w, emic_d_alpha(l_grid_w, energy) + 1e-12,
                     label=f"E={energy} MeV")
for ax, ttl in zip(axes, ["Hiss / 플라즈마권 hiss", "EMIC / EMIC 파동"]):
    ax.axvline(2.8, color="k", ls="--", alpha=0.5)
    ax.set_xlabel("L")
    ax.set_title(ttl)
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel(r"$\langle D_{\alpha\alpha}\rangle$ (1/day)")
axes[0].set_ylim(1e-6, 1e2)
plt.tight_layout()
plt.show()

# Compute lifetimes at L=2.8.
print("Electron lifetime at L=2.8 (hiss-dominated):")
for e in [1, 2, 3, 5]:
    d = hiss_d_alpha(np.array([2.78]), e)[0]
    tau = 1.0 / max(d, 1e-12)
    print(f"  E={e} MeV: tau = {tau:.2f} day")
print("\nEMIC contribution at L=2.8: zero (outside band).")

## Part 6: Summary & Interpretation / 요약 및 해석

**English**
- Carpenter & Anderson (1992) gives L_pp ≈ 2.8 for Kp ≈ 6 — but Baker et al. (2014) note that for the first 20 months (1 Sept 2012 – 1 May 2014) of Van Allen Probes the *plasmapause was generally beyond L ≈ 4* and only occasionally moved as far inward as L ≈ 3.
- The 1D radial-diffusion + sink model used here is a *qualitative* demonstration of how a hiss sink inside the plasmapause produces a sharp inner edge. The actual physics in Baker et al. (Fig. 4) is that for ultrarelativistic electrons (μ = 3000 MeV/G ≈ 5–7 MeV) the radial-transport timescale inside L = 3 *exceeds 1000 days*, while the hiss scattering lifetime is ~100 days. Both timescales are slow, but transport is even slower — so once electrons land near L ≈ 3 they cannot be replenished and decay slowly. The paper frames this as: "weak scattering losses are dominant over even weaker radial diffusion transport" (p. 533).
- Energy selectivity arises because <1 MeV electrons have a hiss lifetime of ~10 days (decaying quickly) while ultrarelativistic electrons live ~100 days but cannot be transported in. Figure 3 a–c shows the inner-edge gradient becoming steeper at higher energies (3.6 → 5.6 → 7.2 MeV).
- The barrier is *not* a fast-loss/slow-transport balance, *not* maintained by VLF transmitters (the paper rejects this; refs. 19, 20), and *not* due to SAA scattering (SAA loss centred at L ≈ 1.5; CSSWE-REPTile >3.8 MeV map confirms separation, Extended Data Fig. 2). It is a natural consequence of slow inward radial diffusion + weak persistent hiss scattering.

**한국어**
- Carpenter & Anderson(1992)은 Kp ≈ 6에서 L_pp ≈ 2.8을 준다. 그러나 Baker 외(2014)는 Van Allen Probes 첫 20개월(2012-09-01 ~ 2014-05-01) 동안 *플라즈마권계면이 대체로 L ≈ 4 바깥*에 위치했으며 가끔만 L ≈ 3까지 안쪽으로 이동했다고 명시한다.
- 본 노트북의 1D 방사형 확산 + 손실 모델은 플라즈마권계면 내부 hiss 손실이 어떻게 날카로운 내측 경계를 만드는지에 대한 *질적* 시연이다. Baker 외(Fig. 4)의 실제 물리는 초상대론적 전자(μ = 3000 MeV/G ≈ 5–7 MeV)의 경우 L = 3 안쪽에서 방사형 수송 시간 척도가 *1000일을 초과*하는 반면, hiss 산란 수명은 약 100일이라는 점이다. 두 시간 척도 모두 느리지만 수송이 더 느리므로, L ≈ 3 부근에 도달한 전자는 보충되지 못하고 천천히 감쇠한다. 논문은 이를 "약한 산란 손실이 더욱 약한 방사형 확산 수송보다 우세"(p. 533)라고 표현한다.
- 에너지 선택성은 <1 MeV 전자의 hiss 수명이 ~10일(빠른 감쇠)인 반면 초상대론적 전자는 ~100일을 살지만 안쪽으로 수송되지 못하기 때문에 발생한다. Figure 3 a–c는 에너지가 높을수록 내측 경계 기울기가 더 가팔라짐(3.6 → 5.6 → 7.2 MeV)을 보여준다.
- Barrier는 *빠른 손실/느린 수송 균형이 아니며*, VLF 송신기에 의해 유지되지 않고(논문이 이를 기각; refs. 19, 20), SAA 산란(L ≈ 1.5 중심; CSSWE-REPTile >3.8 MeV 지도가 분리 확증, Extended Data Fig. 2)에 의한 것도 아니다. 그것은 느린 안쪽 방사형 확산 + 약한 지속적 hiss 산란의 자연스러운 결과이다.